In [8]:
# Step 1: Balanced dataset creation for BattingEdge V8.1
# Run this notebook from BattingEdge_FYP/notebooks

import os, random, shutil
from pathlib import Path
import cv2
import numpy as np

# IMPORTANT: go up one level from /notebooks to project root
ROOT = Path("..")
SRC = ROOT / "data" / "dataset_v7_clean"
DST = ROOT / "data" / "dataset_v8_balanced_videos"

# Use "val" instead of "validation"
SPLITS = ["train", "val", "test"]
CLASSES = ["drive", "pull", "sweep", "cut"]

TARGET_PER_CLASS = 380  # target count per class in train

# Ensure destination structure exists
for split in SPLITS:
    for c in CLASSES:
        (DST / split / c).mkdir(parents=True, exist_ok=True)

def list_videos(p: Path):
    return [f for f in p.rglob("*") if f.suffix.lower() in [".mp4", ".avi", ".mov"]]

def copy_split(src_root: Path, dst_root: Path, split: str):
    for c in CLASSES:
        src_c = src_root / split / c
        dst_c = dst_root / split / c
        dst_c.mkdir(parents=True, exist_ok=True)
        for v in list_videos(src_c):
            shutil.copy2(v, dst_c / v.name)

def read_video(path: Path):
    cap = cv2.VideoCapture(str(path))
    frames = []
    while True:
        ret, f = cap.read()
        if not ret: break
        frames.append(f)
    cap.release()
    return frames

def write_video(frames, out_path: Path, fps=25):
    if not frames: return
    h, w = frames[0].shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(str(out_path), fourcc, fps, (w, h))
    for f in frames:
        out.write(f)
    out.release()

# Augmentations
def aug_flip(frames):
    return [cv2.flip(f, 1) for f in frames]

def aug_brightness(frames, alpha=1.0, beta=15):
    return [cv2.convertScaleAbs(f, alpha=alpha, beta=beta) for f in frames]

def aug_speed(frames, factor=0.9):
    T = len(frames)
    if T < 2: return frames
    idx = np.linspace(0, T-1, max(2, int(T*factor))).astype(int)
    idx = np.clip(idx, 0, T-1)
    return [frames[i] for i in idx]

def aug_temporal_jitter(frames, drop_prob=0.08, dup_prob=0.08):
    out = []
    for f in frames:
        r = random.random()
        if r < drop_prob:
            continue
        out.append(f)
        if r > 1.0 - dup_prob:
            out.append(f)
    return out if len(out) > 0 else frames

AUGS = [
    lambda fr: aug_flip(fr),
    lambda fr: aug_brightness(fr, alpha=0.95, beta=10),
    lambda fr: aug_brightness(fr, alpha=1.05, beta=-5),
    lambda fr: aug_speed(fr, factor=0.85),
    lambda fr: aug_speed(fr, factor=1.15),
    lambda fr: aug_temporal_jitter(fr, drop_prob=0.08, dup_prob=0.08),
]

# 1) Copy val and test as-is
copy_split(SRC, DST, "val")
copy_split(SRC, DST, "test")

# 2) Augment train to reach TARGET_PER_CLASS per class
for c in CLASSES:
    src_cls = list_videos(SRC / "train" / c)
    dst_cls_dir = DST / "train" / c

    # Copy originals first
    for v in src_cls:
        shutil.copy2(v, dst_cls_dir / v.name)

    count = len(src_cls)
    idx_cycle = 0
    while count < TARGET_PER_CLASS and src_cls:
        base = src_cls[idx_cycle % len(src_cls)]
        frames = read_video(base)
        if not frames:
            idx_cycle += 1
            continue
        aug_fn = random.choice(AUGS)
        aug_frames = aug_fn(frames)
        out_name = f"aug_{idx_cycle}_{base.stem}.mp4"
        write_video(aug_frames, dst_cls_dir / out_name, fps=25)
        count += 1
        idx_cycle += 1

print("Augmentation complete. Balanced dataset created at:", DST)

# 3) Print per-class counts after augmentation
def count_split(root: Path, split: str):
    counts = {}
    for c in CLASSES:
        counts[c] = len(list_videos(root / split / c))
    return counts

for split in SPLITS:
    counts = count_split(DST, split)
    print(f"\n{split.upper()} counts:")
    for c in CLASSES:
        print(f"  {c}: {counts[c]}")
    print(f"  TOTAL: {sum(counts.values())}")


Augmentation complete. Balanced dataset created at: ..\data\dataset_v8_balanced_videos

TRAIN counts:
  drive: 381
  pull: 380
  sweep: 380
  cut: 380
  TOTAL: 1521

VAL counts:
  drive: 45
  pull: 47
  sweep: 23
  cut: 35
  TOTAL: 150

TEST counts:
  drive: 73
  pull: 60
  sweep: 25
  cut: 61
  TOTAL: 219
